In [1]:
import pandas as pd

bout_df = pd.read_csv('../output/data_cleaned/sleep_night_bouts_detailed.csv')
bout_df['start_time'] = pd.to_datetime(bout_df['start_time'])
bout_df['end_time'] = pd.to_datetime(bout_df['end_time'])

bout_df

In [2]:
import numpy as np

df_hourly_sleep = bout_df.copy()

df_hourly_sleep['h22'] = np.nan
df_hourly_sleep['h23'] = np.nan
df_hourly_sleep['h00'] = np.nan
df_hourly_sleep['h01'] = np.nan
df_hourly_sleep['h02'] = np.nan
df_hourly_sleep['h03'] = np.nan
df_hourly_sleep['h04'] = np.nan
df_hourly_sleep['h05'] = np.nan

from datetime import datetime, date, time, timedelta
# Function to calculate the overlap in minutes between two time periods
def calculate_overlap(start1, end1, start2, end2):
    latest_start = max(start1, start2)
    earliest_end = min(end1, end2)
    if latest_start < earliest_end:
        return (earliest_end - latest_start).seconds // 60
    return 0

# fixed Time part
time_part_1 = time(22, 00, 00)
time_part_2 = time(6, 00, 00)
    
for index, row in df_hourly_sleep.iterrows():
# construct specific night time   
    date_part_1 = pd.to_datetime(row['sleep_date']) + timedelta(days=-1)
    date_part_2 = pd.to_datetime(row['sleep_date'])
        
    # Combine date and time into a datetime object
    start_of_range = datetime.combine(date_part_1, time_part_1)
    end_of_range = datetime.combine(date_part_2, time_part_2)
    
    # Create a list of hourly slots
    hourly_slots = []
    current_time = start_of_range
    while current_time < end_of_range:
        hourly_slots.append((current_time, current_time + timedelta(hours=1)))
        current_time += timedelta(hours=1)

    
    # Calculate the overlap for each hourly slot
    duration_in_slots = []
    for slot_start, slot_end in hourly_slots:
        overlap_minutes = calculate_overlap(row['start_time'], row['end_time'], slot_start, slot_end)
        duration_in_slots.append(overlap_minutes)
        
    df_hourly_sleep.at[index,'h22'] = duration_in_slots[0]
    df_hourly_sleep.at[index,'h23'] = duration_in_slots[1]
    df_hourly_sleep.at[index,'h00'] = duration_in_slots[2]
    df_hourly_sleep.at[index,'h01'] = duration_in_slots[3]
    df_hourly_sleep.at[index,'h02'] = duration_in_slots[4]
    df_hourly_sleep.at[index,'h03'] = duration_in_slots[5]
    df_hourly_sleep.at[index,'h04'] = duration_in_slots[6]
    df_hourly_sleep.at[index,'h05'] = duration_in_slots[7]
    

In [3]:
df_hourly_sleep.to_csv('../output/sleep_score/hourly_sleep_duration_NIGHT.csv', index=False)
df_hourly_sleep

# calculate sleep regularity

In [6]:
df_reg = df_hourly_sleep[['patient_id', 'sleep_date', 'h22', 'h23', 'h00', 'h01', 'h02', 'h03', 'h04', 'h05']].copy()

df_reg = df_reg.groupby(['patient_id', 'sleep_date'], as_index=False).sum()

columns_to_divide = df_reg.filter(regex='^h')
# get percentage
df_reg[columns_to_divide.columns] = columns_to_divide / 60

df_reg

## cosine similarity

In [20]:
from sklearn.metrics.pairwise import cosine_similarity
# Function to calculate cosine similarity between two vectors
def calculate_cosine_similarity(row1, row2):
    return cosine_similarity([row1], [row2])[0][0]

# Initialize a list to store the similarity scores
similarity_scores = []

for patient_id, group in df_reg.groupby('patient_id'):
    # Loop through the DataFrame rows
    for i in range(len(group)):
        # For the first 5 rows, just output the similarity score as "initial dates"
        if i < 5:
            similarity_scores.append("initial dates")
        else:
            # For each row after the 5th, calculate the average similarity with the previous 5 rows
            avg_similarity = 0
            for j in range(i - 5, i):  # Compare with the previous 5 rows
                row1 = df_reg.iloc[i, 2:10].values  # Current row (vector part)
                row2 = df_reg.iloc[j, 2:10].values  # Previous row (vector part)
                avg_similarity += calculate_cosine_similarity(row1, row2)
            
            avg_similarity /= 5  # Get the average similarity score
            similarity_scores.append(avg_similarity)

# Add the similarity scores to the DataFrame
df_reg['cosine_similarity'] = similarity_scores
df_reg

## Euclidean Distance

In [21]:
from sklearn.metrics.pairwise import euclidean_distances

# Function to calculate Euclidean distance between two vectors
def calculate_euclidean_distance(row1, row2):
    return euclidean_distances([row1], [row2])[0][0]

# Initialize a list to store the similarity scores (distances)
similarity_scores = []

for patient_id, group in df_reg.groupby('patient_id'):
    # Loop through the DataFrame rows
    for i in range(len(group)):
        # For the first 5 rows, just output the similarity score as "initial dates"
        if i < 5:
            similarity_scores.append("initial dates")
        else:
            # For each row after the 5th, calculate the average distance with the previous 5 rows
            avg_distance = 0
            for j in range(i - 5, i):  # Compare with the previous 5 rows
                row1 = df_reg.iloc[i, 2:10].values  # Current row (vector part)
                row2 = df_reg.iloc[j, 2:10].values  # Previous row (vector part)
                avg_distance += calculate_euclidean_distance(row1, row2)
            
            avg_distance /= 5  # Get the average distance score
            similarity_scores.append(avg_distance)

# Add the similarity scores (distances) to the DataFrame
df_reg['euclidean_distance'] = similarity_scores
df_reg


In [27]:
df_sleep_regularity = df_reg[['patient_id', 'sleep_date', 'cosine_similarity', 'euclidean_distance']].copy()

df_sleep_regularity.rename(columns={
    'cosine_similarity': 'sleep_regularity_cs',
    'euclidean_distance': 'sleep_irregularity_dis'
}, inplace=True)

df_sleep_regularity.to_csv('../output/sleep_score/sleep_regularity_score_with Initial Date marks.csv', index=False)

df_sleep_regularity


In [43]:
from sklearn.preprocessing import StandardScaler
df_sleep_regularity['sleep_irregularity_dis'] = pd.to_numeric(df_sleep_regularity['sleep_irregularity_dis'], errors='coerce')

df_sleep_regularity_cleaned = df_sleep_regularity.dropna(subset=['sleep_irregularity_dis']).copy()

scaler = StandardScaler()
df_sleep_regularity_cleaned['sleep_irregularity_dis'] = scaler.fit_transform(df_sleep_regularity_cleaned[['sleep_irregularity_dis']])

df_sleep_regularity_cleaned.to_csv('../output/sleep_score/sleep_regularity_score.csv', index=False)

df_sleep_regularity_cleaned
